In [1]:
# RAG backend setup for interview questions

import os
from dotenv import load_dotenv

import pandas as pd
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEndpointEmbeddings

load_dotenv()  # loads HF / other keys from .ENV or .env

# IMPORTANT: your .ENV must define HF_TOKEN
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not found in environment (.ENV). Please set it before running.")

DATA_PATH = "./questions.csv"  # relative to AI_BACKEND
PERSIST_DIR = "./chroma_db"    # folder where Chroma will store data

print("Current working directory:", os.getcwd())
print("Using data file:", os.path.abspath(DATA_PATH))
print("Chroma persist dir:", os.path.abspath(PERSIST_DIR))


Current working directory: c:\Users\DELL\Desktop\AI-POWERED-INTERVIEW-PREPARATION-COACH\AI_BACKEND
Using data file: c:\Users\DELL\Desktop\AI-POWERED-INTERVIEW-PREPARATION-COACH\AI_BACKEND\questions.csv
Chroma persist dir: c:\Users\DELL\Desktop\AI-POWERED-INTERVIEW-PREPARATION-COACH\AI_BACKEND\chroma_db


In [2]:
# 1) Load questions from CSV via LangChain CSVLoader

loader = CSVLoader(
    file_path=DATA_PATH,
    source_column="question_text",
    metadata_columns=[
        "question_id",
        "role_tag",
        "difficulty_level",
        "subtopic",
        "ideal_answer",
    ],
)
docs = loader.load()
len(docs)

100

In [3]:
# 2) Inspect a sample document (question + metadata)

docs[0].page_content, docs[0].metadata

('question_text: What is Machine Learning?\nsource_ref: ',
 {'source': 'What is Machine Learning?',
  'row': 0,
  'question_id': 'AI_0001',
  'role_tag': 'AI ML Engineer',
  'difficulty_level': 'Easy',
  'subtopic': 'Introduction',
  'ideal_answer': 'ML is a subset of AI where algorithms learn patterns from data to make predictions without being explicitly programmed. AI is the broader field of mimicking human intelligence, while Data Science focuses on extracting insights from data using statistics and ML. Keywords: Subset of AI, Explicit Programming, Pattern Recognition, Predictive Modeling, Data Insights.'})

In [4]:
# 3) Create / load Chroma vector store with Hugging Face Endpoint embeddings

embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",  # or any HF embedding model repo_id
    huggingfacehub_api_token=HF_TOKEN,
)

# If you rerun the notebook, Chroma will reuse the same persistent folder.
vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
)

vectordb.persist()
print("Chroma DB created and persisted at:", os.path.abspath(PERSIST_DIR))

c:\Users\DELL\Desktop\AI-POWERED-INTERVIEW-PREPARATION-COACH\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chroma DB created and persisted at: c:\Users\DELL\Desktop\AI-POWERED-INTERVIEW-PREPARATION-COACH\AI_BACKEND\chroma_db


C:\Users\DELL\AppData\Local\Temp\ipykernel_18288\3177186329.py:15: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


In [5]:
# 4) Test a similarity search query

query = "Explain the difference between supervised and unsupervised learning"
results = vectordb.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}:")
    print("Question:", doc.page_content)
    print("Metadata:", doc.metadata)



Result 1:
Question: question_text: What is Machine Learning?
source_ref: 
Metadata: {'ideal_answer': 'ML is a subset of AI where algorithms learn patterns from data to make predictions without being explicitly programmed. AI is the broader field of mimicking human intelligence, while Data Science focuses on extracting insights from data using statistics and ML. Keywords: Subset of AI, Explicit Programming, Pattern Recognition, Predictive Modeling, Data Insights.', 'question_id': 'AI_0001', 'row': 0, 'difficulty_level': 'Easy', 'subtopic': 'Introduction', 'role_tag': 'AI ML Engineer', 'source': 'What is Machine Learning?'}

Result 2:
Question: question_text: Explain the "Kernel Trick" in Support Vector Machines (SVM).
source_ref: 
Metadata: {'question_id': 'DS_0011', 'subtopic': 'Algorithms', 'difficulty_level': 'Hard', 'role_tag': 'Data Scientist', 'row': 23, 'source': 'Explain the "Kernel Trick" in Support Vector Machines (SVM).', 'ideal_answer': 'The Kernel Trick allows SVM to solve